# P06 — Aprendizaje de secuencia a secuencia con redes neuronales

## 1. Título y paper

**Paper:** *Sequence to Sequence Learning with Neural Networks*  
**Autoría:** Ilya Sutskever, Oriol Vinyals, Quoc V. Le  
**Año y venue:** 2014 · arXiv:1409.3215 · NeurIPS (NIPS) 2014  
**Nivel:** L3 · **Motor:** `seq2seq`  
**Ficha completa:** [`P06_seq2seq`](../../papers/foundational/P06_seq2seq/README.md)

**Hito:** Una única red aprende a mapear secuencias de longitud variable a secuencias de longitud variable, de extremo a extremo.

- [arXiv:1409.3215](https://arxiv.org/abs/1409.3215)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Las redes profundas requerían entradas y salidas de dimensión fija; la traducción automática dependía de sistemas estadísticos con muchas piezas separadas.
2. Ejecutar una implementación mínima de la propuesta: Un LSTM codifica la entrada en un vector de tamaño fijo y otro LSTM lo decodifica token a token; invertir la secuencia fuente mejora el resultado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P03
- Cho et al. (2014), RNN encoder–decoder y GRU


## 4. Intuición

Leer una frase entera, cerrar los ojos, y escribir la traducción solo de memoria. Funciona con frases cortas. Con un párrafo, cuando llegas al final ya no recuerdas cómo empezaba.


## 5. Concepto mínimo

```text
c = encoder(x₁…x_n)                    un único vector de tamaño fijo
p(y₁…y_m) = Π_t p(y_t | y_<t, c)       el decodificador solo ve c
```

Toda la información de la entrada tiene que caber en `c`. La capacidad de `c` no crece con la longitud de la frase: ahí está el cuello de botella.


## 6. Código explicado

El motor codifica secuencias de longitud creciente en un vector fijo y mide cuánto sobrevive del principio.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('seq2seq', seed=7)['result']
print('dimensión del vector de contexto:', r['state_dim'])
for fila in r['bottleneck']:
    print(f"n={fila['length']:>2} · cos(primer token)={fila['cos_primer_token']:+.3f} "
          f"· cos(último)={fila['cos_ultimo_token']:+.3f} · recuperables={fila['tokens_recuperables']}")

## 7. Predicción antes de ejecutar

1. ¿Cómo evolucionará el coseno con el primer token al pasar de n=2 a n=32?
2. ¿Y el coseno con el último token?
3. Si inviertes la secuencia de entrada, ¿a qué extremo beneficias?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for fila in r['bottleneck']:
    brecha = fila['cos_ultimo_token'] - fila['cos_primer_token']
    print(f"n={fila['length']:>2} · brecha último−primero = {brecha:+.3f}")

## 9. Salida interpretable

La brecha crece con la longitud: el vector fijo está dominado por lo último que leyó. Este es exactamente el motivo por el que Sutskever et al. invirtieron la secuencia fuente — así el principio de la frase de entrada queda *cerca* del principio de la de salida.


## 10. Comentario pedagógico

Invertir la entrada es un parche brillante y honesto: no elimina el cuello de botella, reordena qué información se pierde. El siguiente paper eliminará la premisa entera.


## 11. Error o anti-patrón deliberado

Anti-patrón: culpar al tamaño del modelo («faltan parámetros») cuando el problema es estructural.


In [ ]:
print('Diagnóstico incorrecto: «sube la dimensión del estado y se arregla».')
print('Duplicar la dimensión retrasa el problema; no cambia que la capacidad sea CONSTANTE')
print('mientras la longitud de la entrada es VARIABLE.')

## 12. Corrección

Diagnóstico correcto: capacidad constante frente a información creciente.


In [ ]:
diagnostico = {
    'sintoma': 'la calidad cae al crecer la longitud de la entrada',
    'causa_estructural': 'la entrada se comprime en un vector de tamaño fijo',
    'parche_del_paper': 'invertir la secuencia fuente',
    'solucion_de_fondo': 'dejar que el decodificador consulte TODOS los estados (P07)',
}
show(diagnostico)

## 13. Desafío guiado

Sube `state_dim` mentalmente: ¿a partir de qué longitud volvería a fallar? Compruébalo con una versión propia del codificador.


In [ ]:
def codificar(tokens, dim, decaimiento=0.7):
    estado = [0.0] * dim
    for i, t in enumerate(tokens):
        vec = [((i * 7 + d * 3) % 11) / 11 for d in range(dim)]
        estado = [decaimiento * s + (1 - decaimiento) * v for s, v in zip(estado, vec)]
    return estado

for n in (4, 16, 64):
    estado = codificar(list(range(n)), dim=8)
    print(f'n={n:>2} → norma del estado {sum(x*x for x in estado) ** 0.5:.4f}')

## 14. Desafío autónomo

Implementa un seq2seq de juguete para invertir cadenas de longitud variable. Mide la exactitud por longitud (2, 4, 8, 16) con y sin inversión de la entrada. Reporta la curva.


## 15. Evidencia de aprendizaje

Guarda la tabla de cosenos por longitud, la brecha primero-último y el diagnóstico distinguiendo causa estructural de parche.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P06_seq2seq/README.md) · evaluación formal: [`assessments/papers/P06_seq2seq.md`](../../assessments/papers/P06_seq2seq.md)


## 16. Cierre

El cuello de botella tiene nombre y medida. La solución no será un vector más grande, sino dejar de comprimir.


## 17. Conexión con el siguiente hito

- P07

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
